# Making PI and RM maps
## Figures 6-12 in the paper
### A. Ordog, Sept 3, 2024

In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
#from scipy.signal import convolve as scipy_convolve
from astropy.convolution import convolve
import gc
from matplotlib.patches import Circle
from scipy.stats import linregress
import matplotlib as mpl

## Choose thresholds for PI and error in RM

2017 paper (from MSc thesis): dQU is 0.03 K, so dPI is 0.015 K, so 5-sigma threshold is 0.075 K.

PhD thesis: dQU is 0.05 K, so dPI is 0.025 K, so 5-sigma threshold is 0.125 K.

In [ ]:
P_thr   = 0.075 # K
dRM_thr = 5000 # rad/m^2

## Functions 

In [ ]:
def read_files_4channels(directory,stokes,filetype):

    band = ['A','B','C','D']
    data_list = []
    hdr_list = []
    print('Reading in '+stokes+' for filetype: '+filetype)

    for i in range(0,4):
        print('band '+band[i])
        hdu = fits.open(directory+stokes+band[i]+'_'+filetype+'.fits')
        data_list.append(hdu[0].data)

        hdr = fits.Header()
        for card in hdu[0].header.cards:
            if card.keyword.strip() != "":
                hdr.append(card)
        hdr['OBJECT'] = stokes+band[i]+'_'+filetype
        hdr_list.append(hdr)
        #print(repr(hdr))
        #print('-------------------')

    gc.collect()

    return data_list,hdr_list

In [ ]:
def make_landecker_map(data1,data2,hdr,v1max=300,v2max=1,
                       cmap1='RdBu_r',cmap2 = 'viridis',
                       llim = [192,52], blim = [-7,10],filename='test',
                       *args,**kwargs):
    
    aspect = (blim[1]-blim[0])/(llim[0]-llim[1])
    print(aspect)
    
    c = SkyCoord(llim, blim, frame=Galactic, unit="deg")
    fs = 22
    
    fig = plt.figure(figsize=(16,11.3))
    
    plt.subplots_adjust(hspace=0.0,left=0.08, right=0.98, top=0.99, bottom=0.08)
    
    cmap = mpl.colormaps.get_cmap(cmap1)  # viridis is the default colormap for imshow
    cmap.set_bad(color='grey')
    
    ax1  = fig.add_subplot(211, projection=WCS(hdr).celestial)
    im1  = ax1.imshow(data1, origin='lower', vmin=-v1max, vmax=v1max,cmap=cmap)
    ax1.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax1.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    #ax1.set_xticks([125,130,135])
    cbar1 = fig.colorbar(im1, ax=ax1, orientation='vertical',fraction=0.1,pad=0.0,aspect=15)
    cbar1.set_label(r'RM (rad m$^{-2}$)', fontsize=fs)
    cbar1.set_ticks([-200,-100,0,100,200])

    cmap = mpl.colormaps.get_cmap(cmap2)  # viridis is the default colormap for imshow
    cmap.set_bad(color='grey')
   
    ax2  = fig.add_subplot(212, projection=WCS(hdr).celestial)
    im2  = ax2.imshow(data2, origin='lower', vmin=0, vmax=v2max,cmap=cmap)
    ax2.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax2.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    #ax2.set_xticks([125,130,135])
    cbar2 = fig.colorbar(im2, ax=ax2, orientation='vertical',fraction=0.1,pad=0.0,aspect=15)
    cbar2.set_label(r'PI (K)', fontsize=fs)
    cbar2.set_ticks([0,0.1,0.2,0.3,0.4,0.5])
    
    
    ax2.set_xlabel('Galactic Longitude',fontsize=fs)
    fig.text(0.02,0.45,'Galactic Latitude',fontsize=fs,rotation='vertical')
    for ax in [ax1,ax2]:
        ax.tick_params(axis='both', labelsize=fs)
        ax.set_ylabel('  ',fontsize=fs)
        ax.tick_params(axis='both', which='both', width=2, length=6)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
        
    for cbar in [cbar1,cbar2]:
        cbar.ax.tick_params(axis='y', which='both', width=2, length=6)
        cbar.ax.tick_params(labelsize=fs)
        cbar.outline.set_linewidth(2)
    
    #plt.savefig('/home/aordog/CGPS_GMIMS_PLOTS/'+filename+'.pdf')
    plt.savefig('../plots/maps/'+filename+'.pdf')

    return

## Read in the data

In [ ]:
# Read in RM, Pearson R, and standard error in RM:
hdu_RM_CG = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_CG_conv4_regrd.fits')
RM_CG_all = hdu_RM_CG[0].data
RM_CG     = RM_CG_all.copy()
rvalue_CG = hdu_RM_CG[2].data
stderr_CG = hdu_RM_CG[4].data
hdr       = hdu_RM_CG[0].header

hdu_RM_G = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_G_regrd.fits')
RM_G_all = hdu_RM_G[0].data
RM_G     = RM_G_all.copy()
rvalue_G = hdu_RM_G[2].data
stderr_G = hdu_RM_G[4].data

hdu_RM_C = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_C_conv4_regrd.fits')
RM_C_all = hdu_RM_C[0].data
RM_C     = RM_C_all.copy()
rvalue_C = hdu_RM_C[2].data
stderr_C = hdu_RM_C[4].data

# Read in polarised intensity:
hdu_PI_CG = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_CG_conv4_regrd_PI_of_mean.fits')
PI_CG     = hdu_PI_CG[0].data

hdu_PI_G = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_G_regrd_PI_of_mean.fits')
PI_G     = hdu_PI_G[0].data 

hdu_PI_C = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_C_conv4_regrd_PI_of_mean.fits')
PI_C     = hdu_PI_C[0].data 

# Set outside of mosaics to NaN:
RM_CG[RM_CG_all==0.0]     = np.nan
rvalue_CG[RM_CG_all==0.0] = np.nan
stderr_CG[RM_CG_all==0.0] = np.nan
PI_CG[RM_CG_all==0.0]     = np.nan

RM_C[RM_C_all==0.0]     = np.nan
rvalue_C[RM_C_all==0.0] = np.nan
stderr_C[RM_C_all==0.0] = np.nan
PI_C[RM_C_all==0.0]     = np.nan

## Making figures for the paper

In [ ]:
l = WCS(hdr).all_pix2world(range(RM_CG.shape[1]) ,0, 0)[0]
b = WCS(hdr).all_pix2world(0, range(RM_CG.shape[0]), 0)[1]

RM_CG_filt = RM_CG.copy()
RM_G_filt = RM_G.copy()
RM_C_filt = RM_C.copy()
########################################
RM_CG_filt[PI_CG < P_thr] = np.nan
RM_CG_filt[stderr_CG > dRM_thr] = np.nan

RM_G_filt[PI_G < P_thr] = np.nan
RM_G_filt[stderr_G > dRM_thr] = np.nan

RM_C_filt[PI_C < P_thr] = np.nan
RM_C_filt[stderr_C > dRM_thr] = np.nan
########################################

llim_list = [[86,66],[104,84],[122,102],[140,120],[158,138],[176,156],[194,174]]
name_list = ['66_86','84_104','102_122','120_140','138_158','156_176','174_194']

RMmax_list = [300, 300, 200, 200, 200, 200, 200]

#llim_list = [[176,156]]
#name_list = ['156_176']

idx_bad = np.where((l>179.25) & (l<180.75))

RM_CG_filt[:,idx_bad] = np.nan
PI_CG[:,idx_bad] = np.nan
#RM_G_filt[:,idx_bad] = np.nan
#RM_C_filt[:,idx_bad] = np.nan

#make_landecker_map(RM_CG_filt,PI_CG,hdr,v1max=300,v2max=0.2,
#                   cmap1='RdBu_r',cmap2='gist_heat_r',
#                    llim = [86,52], blim = [-6,7.6],filename = 'CGPS_GMIMS_RM_52_86_grey_AO2017')

#make_landecker_map(RM_G_filt,PI_G,hdr,v1max=300,v2max=0.6,
#                   cmap1='RdBu_r',cmap2='gist_heat_r',
#                    llim = [86,52], blim = [-6,7.6],filename = 'GMIMS_RM_52_86_grey_AO2017')

make_landecker_map(RM_C_filt,PI_C,hdr,v1max=300,v2max=0.075,
                   cmap1='RdBu_r',cmap2='gist_heat_r',
                    llim = [86,52], blim = [-6,7.6],filename = 'CGPS_RM_52_86_grey_AO2017')

#for i in range(0,7):
#    make_landecker_map(RM_CG_filt,PI_CG,hdr,v1max=RMmax_list[i],v2max=0.6,
#                       cmap1='RdBu_r',cmap2='gist_heat_r',
#                        llim = llim_list[i], blim = [-3,5],filename = 'CGPS_GMIMS_RM_'+name_list[i]+'_grey_AO2017')

#    make_landecker_map(RM_G_filt,PI_G,hdr,v1max=RMmax_list[i],v2max=0.6,
#                       cmap1='RdBu_r',cmap2='gist_heat_r',
#                        llim = llim_list[i], blim = [-3,5],filename = 'GMIMS_RM_'+name_list[i]+'_grey_AO2017')

#    make_landecker_map(RM_C_filt,PI_C,hdr,v1max=RMmax_list[i],v2max=0.6,
#                       cmap1='RdBu_r',cmap2='gist_heat_r',
#                        llim = llim_list[i], blim = [-3,5],filename = 'CGPS_RM_'+name_list[i]+'_grey_AO2017')
